In [10]:
from crud_python_file import AnimalShelter

username = "aacuser"
password = "YourStrongPassword123"

shelter = AnimalShelter(username, password)

from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()
from dash import html, dcc, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl

import base64
import pandas as pd
import numpy as np
import plotly.express as px


###################################################
# Rescue Queries (ADD THIS SECTION HERE)
###################################################

water_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ]
    },
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}

mountain_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}

tracking_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 20,
        "$lte": 300
    }
}

###################################################

# Retrieve all records
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([

    html.Div(id='hidden-div', style={'display':'none'}),

    html.Center(
        html.B(
            html.H1("Daniel Okonkwo - Grazioso Salvare Dashboard")
        )
    ),
    html.Center(
    html.Img(
        src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={'width': '250px'}
    )
   ),
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label':'Reset', 'value':'RESET'},
            {'label':'Water Rescue', 'value':'WATER'},
            {'label':'Mountain/Wilderness Rescue', 'value':'MOUNTAIN'},
            {'label':'Disaster/Tracking', 'value':'TRACKING'}
        ],
        value='RESET',
        inline=True
    ),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i}
            for i in df.columns
        ],
        data=df.to_dict("records"),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0]
    ),

    html.Br(),

    html.Hr(),

    html.Div(
        id='map-id',
        className='col s12 m6'
    ),
    html.Div(
    id='graph-id'
   )

])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == "WATER":
        data = shelter.read(water_query)

    elif filter_type == "MOUNTAIN":
        data = shelter.read(mountain_query)

    elif filter_type == "TRACKING":
        data = shelter.read(tracking_query)

    else:
        data = shelter.read({})

    df = pd.DataFrame.from_records(data)

    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    return df.to_dict("records")
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Animals by Breed'
            )
        )
    ]
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if not index:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[dff.iloc[row,13], dff.iloc[row,14]],
                    children=[
                        dl.Tooltip(str(dff.iloc[row,4])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row,9]))
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(
    mode="inline",
    debug=False,
    use_reloader=False,
    port=8055
)

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
TypeError: 'NoneType' object is not iterable

